<a href="https://colab.research.google.com/github/peterbabulik/O-LLM/blob/main/O_LLM_RecursiveBracketMatching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ==============================================================================
# OCTONIONIC LARGE LANGUAGE MODEL (O-LLM)
# Benchmarking Non-Associative Attention on Context-Free Grammars (Dyck-3 Language)
# File: O-LLM_RecursiveBracketMatching.py / .ipynb
# Compatible with PyTorch 2.0+ (CUDA GPU & CPU execution)
# ==============================================================================

# %% [1] Environment Setup & Core Dependencies
import random
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Executing on device: {device}")
if torch.cuda.is_available():
    print(f"[*] GPU Model: {torch.cuda.get_device_name(0)}")


# %% [2] Non-Associative Algebraic Engine: Fano Plane Multiplication Tensor
def build_fano_tensor() -> torch.Tensor:
    """
    Constructs the 8x8x8 structural multiplication tensor C_ijk for octonions,
    where: e_i * e_j = sum_k C_ijk * e_k.
    Basis element index 0: e_0 = 1 (Real Identity / Vacuum state).
    Basis elements 1..7: Imaginary units e_1 through e_7 governed by the Fano plane.
    """
    C = torch.zeros((8, 8, 8), dtype=torch.float32)

    # 1. Identity element products: e_0 * e_i = e_i, and e_i * e_0 = e_i
    for i in range(8):
        C[0, i, i] = 1.0
        C[i, 0, i] = 1.0

    # 2. Imaginary squares: e_i * e_i = -e_0 (for i >= 1)
    for i in range(1, 8):
        C[i, i, 0] = -1.0

    # 3. Oriented projective lines (triads) of the Fano plane PG(2, 2)
    fano_triads = [
        (1, 2, 3), (1, 4, 5), (1, 7, 6),
        (2, 4, 6), (2, 5, 7), (3, 4, 7), (3, 6, 5)
    ]

    for i, j, k in fano_triads:
        # Cyclic permutations are positive (+1)
        C[i, j, k] = 1.0;  C[j, k, i] = 1.0;  C[k, i, j] = 1.0
        # Anti-cyclic permutations are negative (-1)
        C[j, i, k] = -1.0; C[k, j, i] = -1.0; C[i, k, j] = -1.0

    return C


class OctonionicAlgebra(nn.Module):
    """
    Batched, vectorized non-associative operations over an 8D octonionic manifold.
    """
    def __init__(self):
        super().__init__()
        self.register_buffer("C", build_fano_tensor())

    def multiply(self, A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
        """Computes element-wise octonionic product A * B."""
        return torch.einsum("...i,...j,ijk->...k", A, B, self.C)

    def associator(self, A: torch.Tensor, B: torch.Tensor, C: torch.Tensor) -> torch.Tensor:
        """
        Computes the ternary Octonionic Associator Bracket:
        [A, B, C] = (A * B) * C - A * (B * C)
        Generates topological vorticity over non-collinear triads.
        """
        AB = self.multiply(A, B)
        AB_C = self.multiply(AB, C)

        BC = self.multiply(B, C)
        A_BC = self.multiply(A, BC)

        res = AB_C - A_BC
        norm = torch.linalg.norm(res, dim=-1, keepdim=True) + 1e-6
        return res / norm


# %% [3] Recursive Octonionic Attention Cell (O-Cell)
class OctonionicCell(nn.Module):
    """
    Recurrent sequence layer with O(1) persistent state memory footprint.
    Processes d_model features grouped into independent 8D octonionic channels.
    """
    def __init__(self, d_model: int):
        super().__init__()
        assert d_model % 8 == 0, "d_model must be a multiple of 8."
        self.d_model = d_model
        self.num_channels = d_model // 8
        self.algebra = OctonionicAlgebra()

        # Learnable chiral gate vector on control line 2a
        gate_init = torch.tensor([1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0] * self.num_channels)
        gate_init = gate_init.view(self.num_channels, 8)
        gate_init = gate_init / torch.linalg.norm(gate_init, dim=-1, keepdim=True)
        self.e_gate = nn.Parameter(gate_init)

        # Asymmetric skip bypass (e3, e5 connection channels)
        self.W_bypass = nn.Parameter(torch.randn(self.num_channels, 8, 8) * 0.05)

        # Gating mixing coefficients
        self.raw_alpha = nn.Parameter(torch.zeros(self.num_channels, 1))
        self.raw_beta = nn.Parameter(torch.zeros(self.num_channels, 1))

    def step(self, h_prev: torch.Tensor, x_t: torch.Tensor) -> torch.Tensor:
        alpha = torch.sigmoid(self.raw_alpha)
        beta = torch.sigmoid(self.raw_beta)

        # 1. Non-associative innovation step via the associator
        gate = self.e_gate.unsqueeze(0)
        delta_h = self.algebra.associator(h_prev, x_t, gate)

        # 2. Linear bypass connection
        bypass = torch.einsum("bci,cij->bcj", x_t, self.W_bypass)

        # 3. State synthesis & Unitary 7-sphere normalization
        h_next = (1.0 - alpha) * h_prev + alpha * delta_h + beta * bypass
        norm = torch.linalg.norm(h_next, dim=-1, keepdim=True) + 1e-6
        return h_next / norm

    def forward(self, x: torch.Tensor, h_init: torch.Tensor = None):
        B, T, D = x.shape
        x_channels = x.view(B, T, self.num_channels, 8)

        if h_init is None:
            # Initialize with the real identity e_0 = 1
            h_curr = torch.zeros((B, self.num_channels, 8), device=x.device, dtype=x.dtype)
            h_curr[..., 0] = 1.0
        else:
            h_curr = h_init

        h_states = []
        for t in range(T):
            h_curr = self.step(h_curr, x_channels[:, t, :, :])
            h_states.append(h_curr)

        out = torch.stack(h_states, dim=1).view(B, T, D)
        return out, h_curr


# %% [4] Language Model with Fano Algebraic Inversion Embeddings
class OctonionicBracketLM(nn.Module):
    """
    Causal Language Model implementing algebraic inversion embeddings.
    Opening brackets apply a positive phase rotation (+e_k);
    Closing brackets apply an exact negative phase inversion (-e_k = e_k*).
    """
    def __init__(self, vocab_size: int, d_model: int = 64, num_layers: int = 2):
        super().__init__()
        assert d_model % 8 == 0
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers

        self.tok_embeddings = nn.Embedding(vocab_size, d_model)

        # Explicit geometric embedding initialization:
        # '(' -> +e1,  ')' -> -e1
        # '[' -> +e2,  ']' -> -e2
        # '{' -> +e4,  '}' -> -e4
        with torch.no_grad():
            self.tok_embeddings.weight.zero_()
            for c in range(d_model // 8):
                base = c * 8
                # Round brackets on channel e1
                self.tok_embeddings.weight[1, base + 1] = 1.0
                self.tok_embeddings.weight[2, base + 1] = -1.0
                # Square brackets on channel e2
                self.tok_embeddings.weight[3, base + 2] = 1.0
                self.tok_embeddings.weight[4, base + 2] = -1.0
                # Curly brackets on channel e4
                self.tok_embeddings.weight[5, base + 4] = 1.0
                self.tok_embeddings.weight[6, base + 4] = -1.0

        self.layers = nn.ModuleList([OctonionicCell(d_model) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)

        # Non-linear phase de-quantization projection head
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, vocab_size)
        )

    def forward(self, idx: torch.Tensor, states: list = None):
        x = self.tok_embeddings(idx)
        new_states = []
        for l_idx, layer in enumerate(self.layers):
            h_prev = states[l_idx] if states is not None else None
            x, h_next = layer(x, h_init=h_prev)
            new_states.append(h_next)
        logits = self.head(self.ln_f(x))
        return logits, new_states


# %% [5] Dyck-3 Grammar Corpus & Targeted Masked Data Generator
BRACKET_PAIRS = {'(': ')', '[': ']', '{': '}'}
OPEN_CHARS = list(BRACKET_PAIRS.keys())
CLOSE_CHARS = list(BRACKET_PAIRS.values())

ALL_CHARS = ['<PAD>'] + OPEN_CHARS + CLOSE_CHARS
vocab_size = len(ALL_CHARS)
char_to_id = {c: i for i, c in enumerate(ALL_CHARS)}
id_to_char = {i: c for i, c in enumerate(ALL_CHARS)}

def generate_clean_dyck(depth=4) -> str:
    """
    Generates a properly nested Dyck-3 sequence of specified depth.
    Ensures strict Last-In-First-Out (LIFO) recursive structure.
    """
    seq = []
    stack = []
    for _ in range(depth):
        op = random.choice(OPEN_CHARS)
        seq.append(op)
        stack.append(BRACKET_PAIRS[op])
    while stack:
        seq.append(stack.pop())
    return "".join(seq)

random.seed(1337)
corpus = [generate_clean_dyck(depth=random.randint(3, 6)) for _ in range(4000)]
print(f"[*] Synthetic Dyck-3 sample generated: {corpus[0]}")

def get_batch_targeted(batch_size=64):
    """
    Prepares training batches with target loss masking focused strictly
    on the closure half (where LIFO stack memory retrieval is verified).
    """
    batch_seqs = [random.choice(corpus) for _ in range(batch_size)]
    max_l = max(len(s) for s in batch_seqs)

    x = torch.zeros((batch_size, max_l), dtype=torch.long)
    y = torch.zeros((batch_size, max_l), dtype=torch.long)
    mask = torch.zeros((batch_size, max_l), dtype=torch.float32)

    for i, s in enumerate(batch_seqs):
        ids = [char_to_id[c] for c in s]
        l = len(ids)
        x[i, :l-1] = torch.tensor(ids[:-1])
        y[i, :l-1] = torch.tensor(ids[1:])
        # Focus loss evaluation solely on the closing sequence
        half = l // 2
        mask[i, half-1 : l-1] = 1.0

    return x.to(device), y.to(device), mask.to(device)


# %% [6] Training Loop: Masked Stack Convergence
model = OctonionicBracketLM(vocab_size=vocab_size, d_model=64, num_layers=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-3, weight_decay=1e-4)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[*] O-LLM initialized with {total_params:,} parameters. Starting training...")

max_steps = 1500
model.train()

for step in range(1, max_steps + 1):
    xb, yb, mask = get_batch_targeted(batch_size=64)
    logits, _ = model(xb)

    # Compute loss exclusively on closing positions
    loss_raw = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1), reduction='none')
    loss = (loss_raw * mask.view(-1)).sum() / (mask.sum() + 1e-6)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if step % 250 == 0 or step == max_steps:
        model.eval()
        with torch.no_grad():
            xv, yv, m_val = get_batch_targeted(batch_size=64)
            v_logits, _ = model(xv)
            preds = torch.argmax(v_logits, dim=-1)
            correct = ((preds == yv) * m_val).sum()
            acc = correct / (m_val.sum() + 1e-6)
        print(f"Step {step:4d}/{max_steps} | Stack Loss: {loss.item():.4f} | Stack Closure Accuracy: {acc.item()*100:.2f}%")
        model.train()

print("[*] Training completed successfully!")


# %% [7] Verification Benchmark: Zero-Shot Recursive Closure Prediction
print("\n" + "="*70)
print("BENCHMARK: ZERO-SHOT RECURSIVE CLOSURE PREDICTION")
print("="*70)

model.eval()
test_prefixes = [
    "({[",          # Expected closure: "]})"
    "[[((",         # Expected closure: "))]]"
    "{[({",         # Expected closure: "})]}"
    "((([{"         # Expected closure: "}]))"
]

with torch.no_grad():
    for prefix in test_prefixes:
        tokens = [char_to_id[c] for c in prefix]
        inp = torch.tensor([tokens], dtype=torch.long).to(device)

        # Compute exact ground truth closing sequence
        stack = [BRACKET_PAIRS[c] for c in prefix]
        expected_closure = "".join(reversed(stack))

        # Autoregressively predict sequence termination
        completed = list(tokens)
        curr = inp
        state = None
        for _ in range(len(expected_closure)):
            logits, state = model(curr, state)
            next_token = torch.argmax(logits[:, -1, :], dim=-1).item()
            completed.append(next_token)
            curr = torch.tensor([[next_token]], dtype=torch.long).to(device)

        full_pred = "".join([id_to_char[i] for i in completed])
        predicted_closure = full_pred[len(prefix):]

        status = "PASSED" if predicted_closure == expected_closure else "FAILED"
        print(f"Prefix: {prefix:<8} | Expected: {expected_closure:<8} | Predicted: {predicted_closure:<8} | [{status}]")

print("="*70)


# %% [8] Save Checkpoint
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': vocab_size,
    'char_to_id': char_to_id,
    'id_to_char': id_to_char,
    'd_model': 64,
    'num_layers': 2
}, "O-LLM_bracket_matching.pt")
print("[*] Model checkpoint successfully saved to 'O-LLM_bracket_matching.pt'")

[*] Executing on device: cuda
[*] GPU Model: Tesla T4
[*] Synthetic Dyck-3 sample generated: {{{([])}}}
[*] O-LLM initialized with 6,375 parameters. Starting training...
Step  250/1500 | Stack Loss: 0.0048 | Stack Closure Accuracy: 99.64%
Step  500/1500 | Stack Loss: 0.0002 | Stack Closure Accuracy: 100.00%
Step  750/1500 | Stack Loss: 0.0002 | Stack Closure Accuracy: 100.00%
Step 1000/1500 | Stack Loss: 0.0000 | Stack Closure Accuracy: 100.00%
Step 1250/1500 | Stack Loss: 0.0000 | Stack Closure Accuracy: 100.00%
Step 1500/1500 | Stack Loss: 0.0000 | Stack Closure Accuracy: 100.00%
[*] Training completed successfully!

BENCHMARK: ZERO-SHOT RECURSIVE CLOSURE PREDICTION
Prefix: ({[      | Expected: ]})      | Predicted: ]})      | [PASSED]
Prefix: [[((     | Expected: ))]]     | Predicted: ))]]     | [PASSED]
Prefix: {[({     | Expected: })]}     | Predicted: })]}     | [PASSED]
Prefix: ((([{    | Expected: }])))    | Predicted: }])))    | [PASSED]
[*] Model checkpoint successfully saved